In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch import nn
import torch
from IPython.display import Markdown
import os

In [2]:
# Setting device on GPU if available, else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cpu


## Category or class names

In [3]:
# Map labels to integers
news_categories=['政治','科技','運動','證卷','產經','娛樂','生活','國際','社會','文化','兩岸']

In [4]:

newslabel_to_id = { cate : i for i, cate in enumerate(news_categories)}

In [5]:
newslabel_to_id

{'政治': 0,
 '科技': 1,
 '運動': 2,
 '證卷': 3,
 '產經': 4,
 '娛樂': 5,
 '生活': 6,
 '國際': 7,
 '社會': 8,
 '文化': 9,
 '兩岸': 10}

In [6]:
id_to_newslabel = { i : cate for i, cate in enumerate(news_categories)}

In [7]:
id_to_newslabel

{0: '政治',
 1: '科技',
 2: '運動',
 3: '證卷',
 4: '產經',
 5: '娛樂',
 6: '生活',
 7: '國際',
 8: '社會',
 9: '文化',
 10: '兩岸'}

# import our custom QwenForClassifier

In [8]:
from custom_qwen_model import QwenForClassifier

# Load

In [9]:
model_id = "Qwen/Qwen2.5-0.5B-instruct"

In [10]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [11]:
# 載入基礎模型
full_model = AutoModelForCausalLM.from_pretrained(model_id).to(device)

In [12]:

hidden_size = full_model.config.hidden_size

#
model_path = "trained_classifier_v1-5epochs-acc0.90"
hidden_size = full_model.config.hidden_size
model_news_classifier = QwenForClassifier(full_model.model, hidden_size, num_labels= len(news_categories))

model_news_classifier.load_model(model_path, device=device)

    
# 移動到指定設備
model_news_classifier = model_news_classifier.to(device)

已載入分類器權重: trained_classifier_v1-5epochs-acc0.90\classifier_weights.pt


# 分類模型怎麼用?

In [13]:

# Function to make predictions
def predict_news_category(text):
    # Tokenize the input text
    inputs = tokenizer(
        text,
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    
    # Get model predictions
    with torch.no_grad():
        outputs = model_news_classifier(**inputs)
    
    # Extract logits and apply softmax to get probabilities
    # logits = outputs.logits
    logits = outputs["logits"]  # 取出 logits
    
    
    probabilities = torch.nn.functional.softmax(logits, dim=-1)
    
    # Get the predicted class (0: negative, 1: positive)
    predicted_class = torch.argmax(probabilities, dim=-1).item()
    
    # Get the class name using id_to_label
    predicted_label = id_to_newslabel[predicted_class]
    
    # Get the confidence score
    confidence = probabilities[0][predicted_class].item()
    
    return {
        "text": text,
        "classification": predicted_label,
        "confidence": round(confidence,2),
        "probabilities": {
            id_to_newslabel[i]: round(prob.item(),2) for i, prob in enumerate(probabilities[0])
        }
    }


In [14]:
%%time
input_text="民進黨與中國國民黨的關係"
predict_news_category(input_text)


We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


CPU times: total: 3.23 s
Wall time: 1.95 s


{'text': '民進黨與中國國民黨的關係',
 'classification': '政治',
 'confidence': 0.9,
 'probabilities': {'政治': 0.9,
  '科技': 0.0,
  '運動': 0.0,
  '證卷': 0.0,
  '產經': 0.0,
  '娛樂': 0.0,
  '生活': 0.01,
  '國際': 0.03,
  '社會': 0.0,
  '文化': 0.01,
  '兩岸': 0.04}}

In [15]:
text = "這個產品品質差，服務更糟糕"
predict_news_category(text)

{'text': '這個產品品質差，服務更糟糕',
 'classification': '生活',
 'confidence': 0.71,
 'probabilities': {'政治': 0.01,
  '科技': 0.02,
  '運動': 0.0,
  '證卷': 0.02,
  '產經': 0.15,
  '娛樂': 0.01,
  '生活': 0.71,
  '國際': 0.02,
  '社會': 0.04,
  '文化': 0.02,
  '兩岸': 0.01}}

In [16]:
text = "這家餐廳的食物美味，環境也很舒適"
predict_news_category(text)

{'text': '這家餐廳的食物美味，環境也很舒適',
 'classification': '生活',
 'confidence': 0.41,
 'probabilities': {'政治': 0.03,
  '科技': 0.01,
  '運動': 0.0,
  '證卷': 0.01,
  '產經': 0.07,
  '娛樂': 0.02,
  '生活': 0.41,
  '國際': 0.07,
  '社會': 0.23,
  '文化': 0.13,
  '兩岸': 0.03}}

# 原始語言模型怎麼用?

In [17]:

#   generated_ids = model.generate(**model_inputs, 
#                                  max_new_tokens=512, 
#                                  do_sample=True, 
#                                  pad_token_id=tokenizer.eos_token_id)

def generate_text(input_prompt):

    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": input_prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(device)

    prompt_length = model_inputs['input_ids'].shape[1]

    generated_ids = full_model.generate(
        model_inputs.input_ids,
        max_new_tokens=512,
        pad_token_id=tokenizer.eos_token_id
    )
    response = tokenizer.decode(generated_ids[0][prompt_length:], skip_special_tokens=True)
    return response


In [18]:
text="給出三個保持健康的提示。"
result = generate_text(text)
Markdown(result)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


1. 均衡飲食，多吃蔬菜水果、粗粮等富含維生素和矿物质的食物。
2. 定期運動，每天至少進行30分鐘的中等至高級有氧運動，如快步走、跑步、游泳等。
3. 內容健康，保持良好的作息習慣，充足的睡眠和適量的飲食，以保持身心健康。

In [19]:
%%time
text="我們如何減少空氣污染？請給幾項重要的建議。"
result = generate_text(text)
Markdown(result)

CPU times: total: 2min 8s
Wall time: 33.7 s


1. 調整能源使用方式：盡量減少車輛的使用，選擇低排放和高效的交通工具，例如電動車、共享車等。
2. 遵守環保法規：遵守當地的環保法規，避免在污染源附近進行任何活動，如開設垃圾填埋場、廢棄物處理站或工業化場地等。
3. 采用節能型產品：使用更節能型的製品，例如LED燈、節能冰箱、節能洗衣机等，以降低能源消耗。
4. 剩余物料管理：將廢棄物和其他可回收物進行妥善處理，減少對環境的影響。
5. 活動中注意安全：在活動過程中注意保護自己，避免吸入有害污染物。
6. 加強公共教育：通過教育提高人們的环保意識，促進社會上更多的綠色生活方式。

以上是一些基本的建議，但需要根據自己的實際情況和環境條件來調整。

In [20]:
%%time
text="亞洲最高的山?"
result = generate_text(text)
Markdown(result)

CPU times: total: 14.2 s
Wall time: 3.93 s


珠穆朗玛峰是亚洲最高的山峰，海拔8,848米。

In [21]:
full_model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((

In [ ]:
# prompt_length=0
# response=tokenizer.decode(generated_ids[0][prompt_length],skip_special_tokens=True)
# response

In [22]:
stop here!

SyntaxError: invalid syntax (2745754519.py, line 1)

# views.py

In [ ]:
from django.shortcuts import render
from django.views.decorators.csrf import csrf_exempt
from django.http import JsonResponse
import json  
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch.nn.functional as F
import markdown

from .custom_qwen_model import QwenForClassifier

# If we don't use GPU, we can set the environment variable to disable it
# os.environ['CUDA_VISIBLE_DEVICES'] = '-1'


# Loading app large language model, news classifier and sentiment classifier

# Setting device on GPU if available, else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)


# 根據設備選擇適當的資料類型
dtype = torch.float16 if device.type == 'cuda' else torch.float32
print(f"使用資料類型: {dtype}")

# Map labels to integers
sentiment_categories=['負面','正面']
sentimentlabel_to_id = { cate : i for i, cate in enumerate(sentiment_categories)}
id_to_sentimentlabel = { i : cate for i, cate in enumerate(sentiment_categories)}

# Convert news category name ('政治','科技','運動',...) into number (0,1,2,...)
news_categories=['政治','科技','運動','證卷','產經','娛樂','生活','國際','社會','文化','兩岸']
newslabel_to_id = { cate : i for i, cate in enumerate(news_categories)}
id_to_newslabel = { i : cate for i, cate in enumerate(news_categories)}

# Tokenizer initialization
model_id = "Qwen/Qwen2.5-0.5B-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
# Model initialization
full_model = AutoModelForCausalLM.from_pretrained(model_id).to(device)
#full_model = AutoModelForCausalLM.from_pretrained(model_id,torch_dtype=dtype).to(device)
hidden_size = full_model.config.hidden_size

# Sentiment classifier model initialization
model_sentiment_classifier = QwenForClassifier(full_model.model, hidden_size, num_labels=len(sentiment_categories))
model_path_sentiment = "app_llm_classifier/trained_models/trained_sentiment_classifier_v4-5epochs-acc0.93"
model_sentiment_classifier.load_model(model_path_sentiment, device=device)
model_sentiment_classifier = model_sentiment_classifier.to(device) # 移動到設備
#model_sentiment_classifier = model_sentiment_classifier.to(device, dtype=dtype) # 移動到設備

# News classifier model initialization
model_news_classifier = QwenForClassifier(full_model.model, hidden_size, num_labels=len(news_categories))
model_path_news = "app_llm_classifier/trained_models/trained_news_classifier_v3-6epochs-acc0.90"
model_news_classifier.load_model(model_path_news, device=device)
#model_news_classifier = model_news_classifier.to(device, dtype=dtype)
model_news_classifier = model_news_classifier.to(device)

# Function to make sentiment predictions
def predict_sentiment(text):
    # Tokenize the input text
    inputs = tokenizer(
        text,
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    
    # Get model predictions
    with torch.no_grad():
        outputs = model_sentiment_classifier(**inputs)
    
    # Extract logits and apply softmax to get probabilities
    logits = outputs["logits"]  
    probabilities = F.softmax(logits, dim=-1)
    
    # Get the predicted class and label
    predicted_class = torch.argmax(probabilities, dim=-1).item()
    predicted_label = id_to_sentimentlabel[predicted_class]
    
    # Get the confidence score
    confidence = probabilities[0][predicted_class].item()
    
    return {
        "text": text,
        "classification": predicted_label,
        "confidence": round(confidence, 2),
        "probabilities": {
            id_to_sentimentlabel[i]: round(prob.item(), 2) for i, prob in enumerate(probabilities[0])
        }
    }

# Function to make news category predictions
def predict_news_category(text):
    # Tokenize the input text
    inputs = tokenizer(
        text,
        max_length=512,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    
    # Get model predictions
    with torch.no_grad():
        outputs = model_news_classifier(**inputs)
    
    # Extract logits and apply softmax to get probabilities
    logits = outputs["logits"]
    probabilities = F.softmax(logits, dim=-1)
    
    # Get the predicted class and label
    predicted_class = torch.argmax(probabilities, dim=-1).item()
    predicted_label = id_to_newslabel[predicted_class]
    
    # Get the confidence score
    confidence = probabilities[0][predicted_class].item()
    
    return {
        "text": text,
        "classification": predicted_label,
        "confidence": round(confidence, 2),
        "probabilities": {
            id_to_newslabel[i]: round(prob.item(), 2) for i, prob in enumerate(probabilities[0])
        }
    }

# Generate text using the LLM
def generate_text(messages):
    """
    Generate response text using the model.
    """
    text_chat_template = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    print("text_chat_templte:", text_chat_template)
    
    model_inputs = tokenizer([text_chat_template], return_tensors="pt").to(device)
    prompt_length = model_inputs['input_ids'].shape[1]

    generated_ids = full_model.generate(
        model_inputs.input_ids,
        attention_mask=model_inputs.attention_mask,
        max_new_tokens=512,
        pad_token_id=tokenizer.eos_token_id
    )
    response = tokenizer.decode(generated_ids[0][prompt_length:], skip_special_tokens=True)
    return response

# Function to handle sentiment prediction
def home_sentiment(request):
    return render(request, "app_llm_classifier/home-sentiment.html")

@csrf_exempt
def api_get_sentiment(request):
    input_text = request.POST.get('input_text')
    print(input_text)
    print(request.content_type)
    print(request.body)

    sentiment_prob = predict_sentiment(input_text)
    return JsonResponse(sentiment_prob)

# Function to handle news category prediction
def home_news_category(request):
    return render(request, "app_llm_classifier/home-news-category.html")

@csrf_exempt
def api_get_news_category(request):
    input_text = request.POST.get('input_text')
    response = predict_news_category(input_text)
    return JsonResponse(response)

# Function to handle text generation using LLM
def home_chatbot(request):
    return render(request, "app_llm_classifier/home-text-generation.html")

@csrf_exempt
def api_get_llm_response(request):
    input_text = request.POST.get('input_text')
    conversation_history_json = request.POST.get('conversation_history')
    
    print("input_text", input_text)
    print("conversation_history_json", conversation_history_json)
    
    # Process conversation history if available
    conversation_history = []
    if conversation_history_json:
        try:
            conversation_history = json.loads(conversation_history_json)
        except json.JSONDecodeError:
            print("Error parsing conversation history JSON")
            conversation_history = []
    
    # Prepare messages for the model
    messages = [
        {"role": "system", "content": "You are a helpful assistant."}
    ]
    
    # Add conversation history if provided
    if conversation_history:
        messages.extend(conversation_history)
    
    # Always add the current user message
    messages.append({"role": "user", "content": input_text})
    
    print("messages:", messages)
    
    # Generate response with the prepared text
    response = generate_text(messages)
    print("response:", response)
    
    # Convert the string response into a dictionary format
    response_dict = {
        "response": response,
        "input": input_text
    }

    return JsonResponse(response_dict)


def model_introduction(request):
    # Read the markdown file
    markdown_file_path = 'app_llm_classifier/markdown-files/model-introduction.md'
    # Read the markdown file
    with open(markdown_file_path, 'r', encoding='utf-8') as f:
        markdown_content = f.read()
    
    # Convert markdown to HTML
    html_content = markdown.markdown(markdown_content, extensions=['fenced_code', 'codehilite'])
    
    # Pass the HTML content to the template
    context = {
        'html_content': html_content
    }
    
    return render(request, "app_llm_classifier/model-introduction.html", context)

print("Loading app large language model, news classifier and sentiment classifier OK.")
